In [1]:
from tqdm.notebook import tqdm
for i in tqdm(range(100)):
    pass

  0%|          | 0/100 [00:00<?, ?it/s]

In [2]:
import mlflow

In [3]:

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("exp1")

2026/03/27 23:46:47 INFO mlflow.tracking.fluent: Experiment with name 'exp1' does not exist. Creating a new experiment.


<Experiment: artifact_location='/mlruns/1', creation_time=1774655207177, experiment_id='1', last_update_time=1774655207177, lifecycle_stage='active', name='exp1', tags={}, workspace='default'>

In [1]:
import polars as pl
data_pl = pl.read_csv(r"../data/train.csv",encoding="shift_jis")

In [2]:
data_new = data_pl.with_columns(
    pl.col("含水率").log().alias("含水率_log")
)

In [4]:
data_new.head(1)

sample number,species number,樹種,含水率,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,9970.62516,9966.76805,9962.91094,9959.05383,9955.19672,9951.33962,9947.48251,9943.6254,9939.76829,9935.91118,9932.05407,9928.19697,9924.33986,9920.48275,9916.62564,9912.76853,9908.91142,9905.05432,9901.19721,9897.3401,9893.48299,9889.62588,9885.76877,9881.91166,9878.05456,9874.19745,9870.34034,…,4134.82018,4130.96307,4127.10596,4123.24886,4119.39175,4115.53464,4111.67753,4107.82042,4103.96331,4100.10621,4096.2491,4092.39199,4088.53488,4084.67777,4080.82066,4076.96356,4073.10645,4069.24934,4065.39223,4061.53512,4057.67801,4053.82091,4049.9638,4046.10669,4042.24958,4038.39247,4034.53536,4030.67826,4026.82115,4022.96404,4019.10693,4015.24982,4011.39271,4007.5356,4003.6785,3999.82139,含水率_log
i64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,1,"""イチョウ""",216.129032,0.41485,0.41465,0.41463,0.41476,0.41481,0.4147,0.41452,0.41427,0.41392,0.41364,0.41354,0.41355,0.41352,0.41337,0.41312,0.41284,0.4127,0.41265,0.41257,0.41242,0.4123,0.41226,0.41218,0.41193,0.41159,0.41129,0.41111,0.41102,0.41096,0.41098,0.41109,0.41116,0.41104,…,1.19969,1.20339,1.20819,1.21138,1.21194,1.21262,1.21483,1.21592,1.21666,1.22043,1.22559,1.22777,1.22751,1.22862,1.23165,1.23629,1.24109,1.24147,1.23927,1.24074,1.24282,1.24082,1.23906,1.24128,1.24471,1.24783,1.25104,1.24925,1.24145,1.2362,1.23384,1.22981,1.22818,1.23087,1.23354,1.23219,5.375876


In [6]:
import polars as pl
import numpy as np
import mlflow
import mlflow.lightgbm
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ===== MLflow設定 =====
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("exp1")

# ===== データ =====
pl_df = data_new  # ← あなたのデータ

# 目的変数
y = pl_df["含水率_log"].to_numpy()

# 不要列除外
drop_cols = ["含水率", "含水率_log", "樹種", "sample number", "species number"]

X = pl_df.drop(drop_cols).to_numpy()

# ===== 分割 =====
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===== 学習 =====
params = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 64,
    "min_data_in_leaf": 5,
    "n_jobs": -1
}

with mlflow.start_run():

    # パラメータ記録
    mlflow.log_params(params)

    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)

    # 予測
    pred = model.predict(X_valid)

    # RMSE（squared=False使えない環境対応）
    rmse = np.sqrt(mean_squared_error(y_valid, pred))

    # ログ
    mlflow.log_metric("rmse", rmse)

    # モデル保存
    mlflow.lightgbm.log_model(model, "model")

    print("RMSE:", rmse)

[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027973 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 396525
[LightGBM] [Info] Number of data points in the train set: 1057, number of used features: 1555
[LightGBM] [Info] Start training from score 3.408297


/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/03/28 00:06:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5


2026/03/28 00:06:39 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RMSE: 0.18669948455892013
🏃 View run entertaining-cub-868 at: http://mlflow:5000/#/experiments/1/runs/0b6458984058453b94948314bb0a87ed
🧪 View experiment at: http://mlflow:5000/#/experiments/1
